# Exploratory single candidate: compact MLP + attested lossless frozen storage
Distinct memory composition after the completed compact-MLP screen: retain the actual native pre-PEFT FP16 tied embedding/head storage only after bit-exact post-PEFT FP32 provenance verification, including signed zero. Native embedding lookup widens to FP32; public non-FP16-AMP head reconstructs the exact FP32 operand transiently. Same Parameter identity and hooks; four narrow source-owned storage attestations, not arbitrary half-weight acceptance. No new embedding/starter kernel credit.
One candidate64-step native task and native public evaluation: `--only solution --repeats 1 --seed 1 --timeout 80`. NO full baseline-training run. The existing bounded one-layer actual HF/NF4/PEFT/SFT fixture additionally requires owned storage/embedding and public-head proxy activation, then authoritative four-owner restoration.
Full-M head, original CE/NF4/RMS math and compact starter-MLP kernels are unchanged. Explicit exploratory policy preserves original thresholds and passed:false warnings; all source/ownership/task/RNG/scaler/output/route gates remain blocking. No performance, paired score or admission claim before target evidence.


In [ ]:
import os, base64, hashlib, json, tempfile
from pathlib import Path
assert os.environ.get('SOUP_REPO') and os.environ.get('SOUP_RESULTS'), 'Requires the deployed prepared job environment; no manual path fallback'
assert Path('/kaggle/working').is_dir(), 'Prepared working directory missing'
MODE = 'full_m_compact_storage_single_v1'
PAYLOAD = {'README.md': 'IyBXZWIgcHJvZmlsZSBnZW5lcmF0aW9uIGtpdCBWMiDigJQgZXZpZGVuY2UgcGFyc2luZyByZXBhaXIgb25seQoKVGhpcyBORVcgZGlyZWN0b3J5IHN1cHBvcnRzIG5ldyBjYW5kaWRhdGUgcmV2aXNpb25zOyBpdCBkb2VzIG5vdCByZXBsYWNlIG9yIG11dGF0ZSBhbnkgc3VibWl0dGVkIG5vdGVib29rLCBzb3VyY2UgWklQLCBhY2NlcHRlZCByZXN1bHQsIG9yIHNoYXJlZCBWMSBmaWxlLiBEbyBub3QgcmVydW4gYW4gdW5jaGFuZ2VkIGNhbmRpZGF0ZSBtZXJlbHkgdG8gcmVwYWlyIHBhcnNpbmc6IG9yaWdpbmFsIHJhdyBsb2dzIHJlbWFpbiBhdmFpbGFibGUgZm9yIGFuYWx5c3QgcmV2aWV3LiBBIHRpbWVvdXQgc3RheXMgZmFpbGVkL2NlbnNvcmVkIHJlZ2FyZGxlc3Mgb2YgcmVjb3ZlcmVkIHJvdXRlIGV2ZW50cy4KCiMjIEludGVyZmFjZQoKYGBgc2gKcHl0aG9uIC93b3Jrc3BhY2Uvc2hhcmVkL3NvdXBfd2ViX3JlYWR5XzIwMjYxMDMwX3YyL2J1aWxkX3Byb2ZpbGVfbm90ZWJvb2sucHkgXAogIC0tbGFuZS1kaXIgL3BhdGgvdG8vTkVXX0xBTkUgLS1wcm9maWxlIC9wYXRoL3RvL3Byb2ZpbGUuanNvbiBcCiAgLS1ub3RlYm9vay1uYW1lIGNhbmRpZGF0ZS5pcHluYgpgYGAKClRoZSBuZXcgbGFuZSBtdXN0IGNvbnRhaW4gYSByZXZpZXdlZCBgY2FuZGlkYXRlLnppcGAgYW5kIGBzb3VyY2VfbWFuaWZlc3QuanNvbmAgKG9yIGBjYW5kaWRhdGUtZmlsZXMuanNvbmApLiBUaGUgYnVpbGRlciByZWZ1c2VzIHRvIG92ZXJ3cml0ZSBgd2ViX2FkYXB0ZXIvYCBvciB0aGUgbm90ZWJvb2suIFNhbWUgc2V2ZW4gcHJvZmlsZSBmaWVsZHMgYXMgVjE6IGBtb2RlYCwgYHRpdGxlYCwgYHByZWZsaWdodGAsIGBwcmVmbGlnaHRfc3RhdHVzYCwgYGtlcm5lbHNgLCBgcm91dGVzYCwgYHJlcXVpcmVzX29yaWdpbmFsX2tlcm5lbHNgIChubyBpbXBsaWNpdCBkZWZhdWx0cykuIEVhY2ggcm91dGUgbm93IGV4cGxpY2l0bHkgcmVxdWlyZXMgYHByZWZpeGAsIGBwb3NpdGl2ZV9maWVsZGAsIGFuZCBlaXRoZXIgYGV2ZW50YCBvciBhIG5vbmVtcHR5IGV4YWN0IGBtYXRjaGAgbWFwLiBPcHRpb25hbCBgbWF0Y2hgIHZhbHVlcyBhcmUgdHlwZS1zZW5zaXRpdmUuIEV4YW1wbGU6CgpgYGBqc29uCnsKICAibW9kZSI6ICJoZWFkX3F1aWV0IiwKICAidGl0bGUiOiAiTmF0aXZlIGhlYWQgcXVpZXQgY29udHJvbCIsCiAgInByZWZsaWdodCI6ICJzb2x1dGlvbi53ZWJfcHJlZmxpZ2h0IiwKICAicHJlZmxpZ2h0X3N0YXR1cyI6ICJQQVNTX05BVElWRV9IRUFEX1BSRUZMSUdIVF9PTkxZIiwKICAia2VybmVscyI6IFtdLAogICJyb3V0ZXMiOiBbCiAgICB7InByZWZpeCI6IlNPVVBfSEVBRF9OQVRJVkUgIiwiZXZlbnQiOiJuYXRpdmVfc3VtX2ZvcndhcmRfcmV0dXJuIiwicG9zaXRpdmVfZmllbGQiOiJjb3VudCIsIm1hdGNoIjp7ImNvdW50Ijo2NH19LAogICAgeyJwcmVmaXgiOiJTT1VQX0hFQURfTkFUSVZFICIsImV2ZW50IjoibmF0aXZlX3N1bV9iYWNrd2FyZF9yZXR1cm4iLCJwb3NpdGl2ZV9maWVsZCI6ImNvdW50IiwibWF0Y2giOnsiY291bnQiOjY0fX0KICBdLAogICJyZXF1aXJlc19vcmlnaW5hbF9rZXJuZWxzIjogZmFsc2UKfQpgYGAKCk5hdGl2ZSB6ZXJvLW93bmVkLWtlcm5lbCBjb250cm9sIHJlbWFpbnMgc3ViamVjdCB0byB1bmNoYW5nZWQgb2ZmaWNpYWwgZWxpZ2liaWxpdHkgcnVsZXMuIElmIHJ1bGVzIGZhaWwsIG5vcm1hbGl6ZWQgY29tcGFyaXNvbiBpcyBudWxsLCBzY29yZSBudWxsLCBhZG1pc3Npb24gVU5LTk9XTi4gTm8gYnlwYXNzLgoKIyMgV2hhdCBjaGFuZ2VzCgpPbmx5IHRoZSBnZW5lcmF0ZWQgcHJvZmlsZSdzIG1ldGFkYXRhIGV2aWRlbmNlIHJlYWRlciBjaGFuZ2VzOiBleGFjdCBsaW5lLWFuY2hvcmVkIHByZWZpeGVzIGFyZSByZWFkIGFmdGVyIHN0cmlwcGluZyB0ZXJtaW5hbCBBTlNJIHNlcXVlbmNlczsgYEpTT05EZWNvZGVyLnJhd19kZWNvZGVgIGFjY2VwdHMgb3JkaW5hcnkgSlNPTiB3aGl0ZXNwYWNlIHdyYXBwaW5nIGFjcm9zcyBsaW5lcy4gSlNPTiBtdXN0IGJlIGFuIG9iamVjdDsgbWFsZm9ybWVkL3RydW5jYXRlZC9vdmVyc2l6ZS9kdXBsaWNhdGUta2V5L25vbmZpbml0ZSBKU09OIGlzIHJlamVjdGVkLiBTY2hlbWEvZXZlbnQvZXhhY3QtbWF0Y2ggYW5kIHBvc2l0aXZlIGZpbml0ZSBudW1lcmljIGNvdW50IHJlcXVpcmVtZW50cyByZW1haW4gbWFuZGF0b3J5LiBCYXJlIHN1YnN0cmluZyBoaXRzLCB3cm9uZyBldmVudHMsIHplcm8vbmVnYXRpdmUvYm9vbCBjb3VudHMgYW5kIGluY29tcGxldGUgcnVucyBjYW5ub3QgcGFzcy4gYG1hdGNoYCBub3cgYWxzbyByZWplY3RzIG1pc21hdGNoZWQgSlNPTiBwcmltaXRpdmUgdHlwZXMgKGUuZy4gaW50ZWdlcjY0IHZlcnN1cyBzdHJpbmciNjQiIG9yIGZsb2F0NjQuMCkuCgpBY3R1YWwgc291cmNlLWJvdW5kIEZ1c2lvbkIgbG9ncyBkZW1vbnN0cmF0ZWQgd2h5IHRoaXMgaXMgbmVlZGVkOiBWMSBzaW5nbGUtbGluZSBwYXJzaW5nIHNhdyBvbmx5IGluc3RhbGxhdGlvbiBldmVudHMsIGJlY2F1c2UgUmljaCBpbnNlcnRlZCBjdXJzb3IgY29udHJvbHMgYmVmb3JlIHJ1bnRpbWUgcHJlZml4ZXMgYW5kIHdyYXBwZWQgdGhlIEpTT04uIFYyIHJlY292ZXJzIHRob3NlIHJlY29yZHMgZm9yIGFuYWx5c2lzOyBpdCAqKmRvZXMgbm90KiogY29udmVydCB0aGF0ODAtc2Vjb25kIHRpbWVkLW91dCBydW4gaW50byBjb21wbGV0aW9uIG9yIGEgbWVhc3VyZW1lbnQuCgojIyBXaGF0IHN0YXlzIHVuY2hhbmdlZAoKLSBObyBudW1lcmljYWwga2VybmVsLCBtb2RlbCwgdGFzaywgcHJlY2lzaW9uLCBvcHRpbWl6ZXIsIGRhdGEsIExvUkEsIGhhcm5lc3MsIHByb3RlY3RlZCBzb3VyY2UgaGFzaCwgcnVudGltZSwgY2FjaGUsIHRyYWluaW5nIGR1cmF0aW9uLCBjYW5jZWxsYXRpb24gb3IgZXhwb3J0IGJlaGF2aW9yIGNoYW5nZXMuCi0gQWxsIG9yaWdpbmFsIHNoYXJlZCBydW5uZXIgZnVuY3Rpb25zIGFyZSBBU1QtaWRlbnRpY2FsIGluIHRoZSBWMiBzZWVkIHJ1bm5lcjsgb25seSBhbiBpbXBvcnQgb2YgdGhlIGV2aWRlbmNlIGhlbHBlciBpcyBhZGRlZC4gR2VuZXJhdGVkIHN0YW5kYWxvbmUgYG5vcm1hbGl6ZWAgc3BlY2lhbGl6ZXMgbWV0YWRhdGEgcm91dGVzIGFzIGJlZm9yZS4gR2VuZXJhdGVkIGBtYWluYCBkaWZmZXJzIG9ubHkgaW4gQ0xJIG1vZGUgbWV0YWRhdGE7IGl0cyBleGVjdXRpb24gYm9keSBpcyB1bmNoYW5nZWQuCi0gU2FtZSBgU09VUF9SRVBPYC9gU09VUF9SRVNVTFRTYCwgYC9rYWdnbGUvd29ya2luZ2AsIGdpdGxlc3MgZXhhY3Qtc291cmNlIHZlcmlmaWNhdGlvbiwgcnVudGltZS9pZGxlL3ByZWZsaWdodCBjaGlsZCBnYXRlcywgdW5jaGFuZ2VkIG9mZmljaWFsIHBhaXJlZDY0LXN0ZXAgdGFzayBhbmQgKio4MC1zZWNvbmQgY2FwIHBlciBuYXRpdmUgdHJhaW5pbmcgcnVuKiouCi0gU2FtZSBwaWRmZC9zdWJyZWFwZXIgZ3VhcmRpYW4sIGluaGVyaXRlZCBsb2NrLCBxdWFyYW50aW5lLCB1bmNlcnRhaW4tY2xlYW51cCBoYW5kbGluZywgb3V0cHV0IGNvbXBsZXRlbmVzcyBnYXRlcywgZml4ZWQ1Nk1pQiBleHBvcnQgYnVkZ2V0LCBzY29yZSBudWxsIGFuZCBhZG1pc3Npb24gVU5LTk9XTi4KLSBWMSBgMDBfYmFzZWxpbmUuaXB5bmJgIGlzIHN0b3JlZCBvbmx5IGFzIGB3b3JrZmxvd19zZWVkLmlweW5iYCB0byByZXVzZSBpdHMgcmV2aWV3ZWQgY2VsbCB3b3JrZmxvdy4gSXQgaXMgbm90IGEgbmV3IHJ1bm5hYmxlIGJhc2VsaW5lIHJlbGVhc2UgYW5kIGlzIG5ldmVyIHN1Ym1pdHRlZCBieSB0aGlzIGtpdC4gUGF5bG9hZHMgYXJlIGZ1bGx5IHJlcGxhY2VkIGluIGdlbmVyYXRlZCBub3RlYm9va3MuCi0gVjIgaXMgZm9yIE5FVyBzb3VyY2UvY2FuZGlkYXRlIHZhcmlhbnRzLiBObyBwcmVmbGlnaHQgYnlwYXNzLCBpbnN0YWxsL2Rvd25sb2FkLCBhdXRoZW50aWNhdGVkIHJlcXVlc3Qgb3Igc3VibWlzc2lvbiBvY2N1cnMgaW4gZ2VuZXJhdGlvbi9DUFUgdGVzdHMuCgpUaGUgaW1wb3J0LW9ubHkgVjIgc2VlZCBydW5uZXIgcmV0YWlucyB0aGUgb2xkIEZ1c2lvbiBBL0IgbWV0YWRhdGEgYnJhbmNoZXMgc29sZWx5IGFzIGluZXJ0IHNlZWQgY29tcGF0aWJpbGl0eTsgdGhlIGJ1aWxkZXIgcHJvaGliaXRzIHRob3NlIG1vZGVzIGFuZCBkb2VzIG5vdCBpc3N1ZSBuZXcwMC8wMS8wMiBub3RlYm9va3MuIE9ubHkgZ2VuZXJhdGVkIG5ldyBwcm9maWxlcyB1c2UgdGhlIHJlcGFpcmVkIHJlYWRlci4KClNlZSBgdmFsaWRhdGlvbi92Ml9yZXN1bHRzLmpzb25gLCBgdmFsaWRhdGlvbi92Ml9jaGVja3MubG9nYCwgYW5kIGB2YWxpZGF0aW9uL2ltbXV0YWJsZV92MV9oYXNoZXMuanNvbmAuIEdQVSBwZXJmb3JtYW5jZSwgdGFyZ2V0IGV4ZWN1dGlvbiwgcXVhbGl0eSBhbmQgYWRtaXNzaW9uIGFyZSBub3QgZXN0YWJsaXNoZWQgYnkgdGhlc2UgcGFyc2VyIGZpeHR1cmVzLgo=', 'runtime_api_reference.json': 'ewogICJwcm92ZW5hbmNlIjogIlBhcmVudCBhZ2VudCBhdXRoZW50aWNhdGVkIHJlYWQtb25seSBydW50aW1lIEdFVCwgdmVyaWZpZWQgVExTLCAyMDI2LTEwLTMwIGFyb3VuZCAyMDo1MCBVVEM7IG5vIHJlcXVlc3QgYnkgdGhpcyBhZGFwdGVyIGF1dGhvciBhbmQgbm8gR1BVIGpvYiIsCiAgImltYWdlIjogInNoYTI1NjpkMjgxMzlmNjljNTZhYzg5N2IyNjQ4N2YwZjdkODIzNmQ3MDFmMDMwMjE0MzQxYjUxMTNiNDcxNDI2N2MzZWM5IiwKICAiZ3B1IjogIlRlc2xhIFQ0IiwgInZyYW1fbWliIjogMTUzNjAsICJkcml2ZXIiOiAiNTgwLjE3OC4wNCIsCiAgInB5dGhvbiI6ICIzLjEzLjE1IiwgInRvcmNoIjogIjIuMTEuMCtjdTEyOCIsICJjdWRhX2J1aWxkIjogIjEyLjgiLCAidHJpdG9uIjogIjMuNi4wIiwgIm51bXB5IjogIjIuMy41IiwKICAiY3B1X2FmZmluaXR5IjogWzAsMSwyLDNdLCAiY2dyb3VwX3JhbV9naWIiOiAzMCwgImhvc3RfcmVwb3J0ZWRfcmFtX2dpYiI6IDI5LjM3LAogICJTT1VQX1JFUE8iOiAiL2thZ2dsZS93b3JraW5nL3JlcG8iLCAiU09VUF9SRVNVTFRTIjogIi9rYWdnbGUvd29ya2luZy9yZXN1bHRzIiwKICAib2ZmaWNpYWxfY29tbWl0IjogImM3YzJmY2QxNTk3MmNlOGI4OThkOGQzZmY3MzJlZTE5MmM4MTdjZWIiLAogICJ0cmFpbmluZ190aW1lb3V0X3MiOiA4MCwgInB1YmxpY19xdWFsaXR5X2RlbHRhX2xpbWl0X3BjdCI6IDAuNSwgInBlYWtfcmVncmVzc2lvbl9saW1pdF9wY3QiOiAxMCwKICAiY29uY3VycmVuY3kiOiAxLCAibmV0d29yayI6ICJub25lIiwgImF1dGhlbnRpY2F0ZWRfdXBsb2FkX2F1dG9fc3RhcnRzIjogdHJ1ZSwKICAibm90ZWJvb2tfbGltaXRfYnl0ZXMiOiA4Mzg4NjA4LCAiZXhwb3J0X2xpbWl0X2J5dGVzIjogNjcxMDg4NjQsCiAgImRlZmF1bHRfam9iX3RpbWVvdXRfcyI6IDM2MDAsICJtYXhfam9iX3RpbWVvdXRfcyI6IDE0NDAwLAogICJkZXZpY2VfZXhlY3V0aW9uX3Byb29mIjogZmFsc2UsICJncHVfam9iX3J1bl9ieV91cyI6IGZhbHNlCn0K', 'owned_process_tree.py': 'IyBTaGFyZWQgdGFzayBvcmNoZXN0cmF0aW9uIGZyb20gdGhlIExvUkEgbGFuZTsgbm8ga2VybmVsIG93bmVyc2hpcCBjcmVkaXQuCiIiIkxpbnV4LW9ubHkgZGVkaWNhdGVkLXJ1bm5lciBjbGVhbnVwOyBubyBHUFUgaW1wb3J0cyBvciBiYWNrZ3JvdW5kIGFjdGl2aXR5LgoKT3JpZ2luYWwgdGFzay1vcmNoZXN0cmF0aW9uIGhlbHBlci4gTm90IHBhcnQgb2Ygc29sdXRpb24vIG9yIG93bi1rZXJuZWwgY3JlZGl0LgoiIiIKaW1wb3J0IGN0eXBlcwppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaWduYWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKCmRlZiBwcmVwYXJlKCk6CiAgICAiIiJGYWlsIGJlZm9yZSBHUFUgd29yayBpZiByYWNlLXNhZmUgcHJvY2VzcyBvd25lcnNoaXAgaXMgdW5hdmFpbGFibGUuIiIiCiAgICBpZiBub3QgaGFzYXR0cihvcywncGlkZmRfb3BlbicpIG9yIG5vdCBoYXNhdHRyKHNpZ25hbCwncGlkZmRfc2VuZF9zaWduYWwnKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0xpbnV4IHBpZGZkIHByb2Nlc3Mgb3duZXJzaGlwIHJlcXVpcmVkJykKICAgIGZkPW9zLnBpZGZkX29wZW4ob3MuZ2V0cGlkKCkpO29zLmNsb3NlKGZkKQogICAgbGliYz1jdHlwZXMuQ0RMTChOb25lLHVzZV9lcnJubz1UcnVlKQogICAgaWYgbGliYy5wcmN0bCgzNiwxLDAsMCwwKSE9MDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0Nhbm5vdCBlbmFibGUgY2hpbGQgc3VicmVhcGVyOyBubyBHUFUgam9iIHN0YXJ0ZWQnKQogICAgZW5hYmxlZD1jdHlwZXMuY19pbnQoMCkKICAgIGlmIGxpYmMucHJjdGwoMzcsY3R5cGVzLmJ5cmVmKGVuYWJsZWQpLDAsMCwwKSE9MCBvciBlbmFibGVkLnZhbHVlIT0xOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQ2hpbGQgc3VicmVhcGVyIHZlcmlmaWNhdGlvbiBmYWlsZWQnKQoKCmRlZiBpZGVudGl0eShwaWQpOgogICAgdHJ5OgogICAgICAgIGZpZWxkcz1QYXRoKGYnL3Byb2Mve3BpZH0vc3RhdCcpLnJlYWRfdGV4dCgpLnJzcGxpdCgnKScsMSlbMV0uc3BsaXQoKQogICAgICAgIHJldHVybiBpbnQoZmllbGRzWzFdKSxpbnQoZmllbGRzWzE5XSksZmllbGRzWzBdCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcixJbmRleEVycm9yKTpyZXR1cm4gTm9uZQoKCmRlZiBkZXNjZW5kYW50cygpOgogICAgcmVjb3Jkcz17fQogICAgZm9yIHAgaW4gUGF0aCgnL3Byb2MnKS5pdGVyZGlyKCk6CiAgICAgICAgaWYgcC5uYW1lLmlzZGlnaXQoKToKICAgICAgICAgICAgcmVjPWlkZW50aXR5KGludChwLm5hbWUpKQogICAgICAgICAgICBpZiByZWMgaXMgbm90IE5vbmU6cmVjb3Jkc1tpbnQocC5uYW1lKV09cmVjCiAgICBvd25lZD17b3MuZ2V0cGlkKCl9CiAgICB3aGlsZSBUcnVlOgogICAgICAgIGFkZD17cGlkIGZvciBwaWQsKHBhcmVudCxfLF8pIGluIHJlY29yZHMuaXRlbXMoKSBpZiBwYXJlbnQgaW4gb3duZWR9LW93bmVkCiAgICAgICAgaWYgbm90IGFkZDpicmVhawogICAgICAgIG93bmVkLnVwZGF0ZShhZGQpCiAgICByZXR1cm4ge3BpZDpyZWNvcmRzW3BpZF0gZm9yIHBpZCBpbiBvd25lZCBpZiBwaWQhPW9zLmdldHBpZCgpIGFuZCBwaWQgaW4gcmVjb3Jkc30KCgpkZWYgc2lnbmFsX2V4YWN0KHBpZCxiaXJ0aCxzaWcpOgogICAgIiIiTmV2ZXIgc2lnbmFsIGEgcmVjeWNsZWQgUElELiBUaGUgRkQgcGlucyBpZGVudGl0eSB0aHJvdWdoIHRoZSBzZW5kLiIiIgogICAgdHJ5OmZkPW9zLnBpZGZkX29wZW4ocGlkKQogICAgZXhjZXB0IFByb2Nlc3NMb29rdXBFcnJvcjpyZXR1cm4KICAgIHRyeToKICAgICAgICBub3c9aWRlbnRpdHkocGlkKQogICAgICAgIGlmIG5vdyBpcyBub3QgTm9uZSBhbmQgbm93WzFdPT1iaXJ0aDoKICAgICAgICAgICAgdHJ5OnNpZ25hbC5waWRmZF9zZW5kX3NpZ25hbChmZCxzaWcsTm9uZSwwKQogICAgICAgICAgICBleGNlcHQgUHJvY2Vzc0xvb2t1cEVycm9yOnBhc3MKICAgIGZpbmFsbHk6b3MuY2xvc2UoZmQpCgoKZGVmIF90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWcpOgogICAgdHJ5OnNpZ25hbF9leGFjdChwaWQsYmlydGgsc2lnKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgICMgRG8gbm90IHVud2luZC9yZWxlYXNlIEdQVSBvd25lcnNoaXAgd2hlbiBjbGVhbnVwIGlzIHVuY2VydGFpbi4KICAgICAgICBxdWFyYW50aW5lKCdDbGVhbnVwIHNpZ25hbCBmYWlsZWQ6ICcrcmVwcihleGMpKQogICAgICAgIHByaW50KCdTT1VQX0NMRUFOVVBfUEVORElORyAnK3JlcHIoZXhjKSxmaWxlPXN5cy5zdGRlcnIsZmx1c2g9VHJ1ZSkKCgpkZWYgY2xlYW51cChwcm9jKToKICAgIHNhdmVkPXtzOnNpZ25hbC5zaWduYWwocyxzaWduYWwuU0lHX0lHTikgZm9yIHMgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pfQogICAgdHJ5OgogICAgICAgIGZyb3plbj1zZXQoKQogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIG9ic2VydmVkPWRlc2NlbmRhbnRzKCkKICAgICAgICAgICAgbmV3PXsocGlkLHZbMV0pIGZvciBwaWQsdiBpbiBvYnNlcnZlZC5pdGVtcygpfS1mcm96ZW4KICAgICAgICAgICAgaWYgbm90IG5ldzpicmVhawogICAgICAgICAgICBmb3IgcGlkLGJpcnRoIGluIG5ldzpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR1NUT1ApCiAgICAgICAgICAgIGZyb3plbi51cGRhdGUobmV3KQogICAgICAgIGZvciBwaWQsYmlydGggaW4gZnJvemVuOl90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWduYWwuU0lHS0lMTCkKICAgICAgICBwcm9jLndhaXQoKQogICAgICAgICMgTm8gdGltZW91dC1iYXNlZCByZWxlYXNlOiBpZiBhIEdQVSBjaGlsZCBpcyBzdHVjayBpbiBrZXJuZWwgZXhpdCwKICAgICAgICAjIHJldGFpbiB0aGlzIHN1cGVydmlzb3IgYW5kIGl0cyBsb2NrIHVudGlsIGV4aXQgaXMgYWN0dWFsbHkgb2JzZXJ2ZWQuCiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBwaWQsXz1vcy53YWl0cGlkKC0xLG9zLldOT0hBTkcpCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IHBpZDpicmVhawogICAgICAgICAgICAgICAgZXhjZXB0IENoaWxkUHJvY2Vzc0Vycm9yOmJyZWFrCiAgICAgICAgICAgIHJlbWFpbmluZz1kZXNjZW5kYW50cygpCiAgICAgICAgICAgIGlmIG5vdCByZW1haW5pbmc6YnJlYWsKICAgICAgICAgICAgZm9yIHBpZCwoXyxiaXJ0aCxfKSBpbiByZW1haW5pbmcuaXRlbXMoKTpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR0tJTEwpCiAgICAgICAgICAgIHRpbWUuc2xlZXAoLjA1KQogICAgZmluYWxseToKICAgICAgICBmb3Igc2lnLGhhbmRsZXIgaW4gc2F2ZWQuaXRlbXMoKTpzaWduYWwuc2lnbmFsKHNpZyxoYW5kbGVyKQoKCmRlZiBydW4oYXJndiwqLGN3ZCxlbnYsc3Rkb3V0LHRpbWVvdXQ9Tm9uZSxzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQpOgogICAgIiIiUnVuIGluIGRlZGljYXRlZCBzdXBlcnZpc29yIG9ubHk7IGNsZWFudXAgbmVzdGVkIHNlc3Npb25zIG9uIGFueSBmYWlsdXJlLiIiIgogICAgcHJvYz1zdWJwcm9jZXNzLlBvcGVuKGFyZ3YsY3dkPWN3ZCxlbnY9ZW52LHN0ZG91dD1zdGRvdXQsc3RkZXJyPXN0ZGVycixzdGFydF9uZXdfc2Vzc2lvbj1UcnVlKQogICAgdHJ5OgogICAgICAgIGNvZGU9cHJvYy53YWl0KHRpbWVvdXQ9dGltZW91dCkKICAgICAgICBpZiBkZXNjZW5kYW50cygpOgogICAgICAgICAgICBxdWFyYW50aW5lKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50cycpCiAgICAgICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50czsgY2xlYW5lZCBhbmQgcmVqZWN0ZWQnKQogICAgICAgIHJldHVybiBjb2RlCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcXVhcmFudGluZSgnT3V0ZXIgdGltZW91dC9pbnRlcnJ1cHRpb24vZXJyb3I6ICcrcmVwcihleGMpKQogICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICByYWlzZQoKCmRlZiBpbnN0YWxsX3NpZ25hbF9oYW5kbGVycygpOgogICAgZGVmIGludGVycnVwdGVkKHNpZ251bSxmcmFtZSk6CiAgICAgICAgcXVhcmFudGluZSgnUnVubmVyIGludGVycnVwdGVkIGJ5IHNpZ25hbCAnK3N0cihzaWdudW0pKQogICAgICAgIHJhaXNlIEludGVycnVwdGVkRXJyb3IoJ1J1bm5lciBpbnRlcnJ1cHRlZCBieSBzaWduYWwgJytzdHIoc2lnbnVtKSkKICAgIGZvciBzaWcgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pOnNpZ25hbC5zaWduYWwoc2lnLGludGVycnVwdGVkKQoKIyBBIGNyYXNoLXN1cnZpdmluZyBtYXJrZXIgaXMgcGxhY2VkIGJlZm9yZSBhbnkgR1BVIHN1YnByb2Nlc3MuIE9ubHkgaXRzIGV4YWN0CiMgb3duZXIgbWF5IGNsZWFyIGl0IG9uIGEgY2xlYW4sIG5vbi1pbnRlcnJ1cHRlZCBleGl0IHdpdGggbm8gbGl2ZSBkZXNjZW5kYW50cy4KIyBJbnRlcnJ1cHRlZC90aW1lZC1vdXQgd29yayBsZWF2ZXMgaXQgZm9yIGV4cGxpY2l0IG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uLgpfTUFSS0VSID0gTm9uZQpfVE9LRU4gPSBOb25lCl9LRUVQX01BUktFUiA9IEZhbHNlCgoKZGVmIGNoZWNrX3F1YXJhbnRpbmUocGF0aCk6CiAgICBpZiBQYXRoKHBhdGgpLmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVMCBxdWFyYW50aW5lZDogcmVjb25jaWxlICcrc3RyKHBhdGgpKycgYmVmb3JlIGFub3RoZXIgcnVuJykKCgpkZWYgYmVnaW5fcXVhcmFudGluZShwYXRoKToKICAgIGdsb2JhbCBfTUFSS0VSLF9UT0tFTixfS0VFUF9NQVJLRVIKICAgIGltcG9ydCBqc29uLHV1aWQKICAgIF9NQVJLRVI9UGF0aChwYXRoKTtfVE9LRU49dXVpZC51dWlkNCgpLmhleDtfS0VFUF9NQVJLRVI9RmFsc2UKICAgIGZkPW9zLm9wZW4oc3RyKF9NQVJLRVIpLG9zLk9fV1JPTkxZfG9zLk9fQ1JFQVR8b3MuT19FWENMfG9zLk9fTk9GT0xMT1csMG82MDApCiAgICB3aXRoIG9zLmZkb3BlbihmZCwndycpIGFzIG91dDoKICAgICAgICBqc29uLmR1bXAoeydvd25lcl90b2tlbic6X1RPS0VOLCdvd25lcl9waWQnOm9zLmdldHBpZCgpLCdzdGF0dXMnOidPV05FRF9HUFVfV09SS19JTl9QUk9HUkVTUyd9LG91dCkKICAgICAgICBvdXQuZmx1c2goKTtvcy5mc3luYyhvdXQuZmlsZW5vKCkpCgoKZGVmIHF1YXJhbnRpbmUocmVhc29uKToKICAgIGdsb2JhbCBfS0VFUF9NQVJLRVIKICAgIF9LRUVQX01BUktFUj1UcnVlCiAgICBpZiBfTUFSS0VSIGlzIE5vbmU6cmV0dXJuCiAgICBpbXBvcnQganNvbgogICAgdHJ5OnZhbHVlPWpzb24ubG9hZHMoX01BUktFUi5yZWFkX3RleHQoKSkKICAgIGV4Y2VwdCAoT1NFcnJvcixWYWx1ZUVycm9yKTpyZXR1cm4KICAgIGlmIHZhbHVlLmdldCgnb3duZXJfdG9rZW4nKSE9X1RPS0VOOnJldHVybgogICAgdmFsdWUudXBkYXRlKHN0YXR1cz0nQkxPQ0tFRF9PUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCcscmVhc29uPXN0cihyZWFzb24pKQogICAgX01BUktFUi53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUpKydcbicpCgoKZGVmIGZpbmlzaF9xdWFyYW50aW5lKCk6CiAgICBpZiBfTUFSS0VSIGlzIE5vbmUgb3IgX0tFRVBfTUFSS0VSOnJldHVybgogICAgaWYgZGVzY2VuZGFudHMoKToKICAgICAgICBxdWFyYW50aW5lKCdMaXZlIG93bmVkIGRlc2NlbmRhbnRzIHJlbWFpbiBhdCBzdXBlcnZpc29yIGV4aXQnKQogICAgICAgIHJldHVybgogICAgaW1wb3J0IGpzb24KICAgIHRyeTp2YWx1ZT1qc29uLmxvYWRzKF9NQVJLRVIucmVhZF90ZXh0KCkpCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcik6cmV0dXJuCiAgICBpZiB2YWx1ZS5nZXQoJ293bmVyX3Rva2VuJyk9PV9UT0tFTjpfTUFSS0VSLnVubGluaygpCg==', 'runtime_probe.py': 'IiIiUmVhZC1vbmx5LCBzaG9ydC1saXZlZCBkZXBsb3llZC10ZW1wbGF0ZSBnYXRlLiBOZXZlciBwcmludHMgc2VjcmV0IHZhbHVlcy4iIiIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YSBhcyBtZXRhZGF0YSxqc29uLG9zLHBsYXRmb3JtLHN1YnByb2Nlc3Msc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApFWFBFQ1RFRD17J3RvcmNoJzonMi4xMS4wK2N1MTI4JywndHJpdG9uJzonMy42LjAnLCdudW1weSc6JzIuMy41Jywnc291cC1jbGknOicwLjc3LjAnLCd0cmFuc2Zvcm1lcnMnOic1LjE2LjEnLCdwZWZ0JzonMC4yMC4wJywndHJsJzonMC4yOS4xJywnYml0c2FuZGJ5dGVzJzonMC41MC4yJywnYWNjZWxlcmF0ZSc6JzEuMTQuMCcsJ2RhdGFzZXRzJzonNC44LjUnfQoKZGVmIG1haW4oKToKIG91dD1QYXRoKHN5cy5hcmd2WzFdKTtyPXsncGFzc2VkJzpGYWxzZSwnZ3B1X3dvcmtsb2FkX2V4ZWN1dGVkJzpGYWxzZSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfQogdHJ5OgogIHZlcnNpb25zPXtrOm1ldGFkYXRhLnZlcnNpb24oaykgZm9yIGsgaW4gRVhQRUNURUR9CiAgci51cGRhdGUocHl0aG9uPXBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCkscGFja2FnZXM9dmVyc2lvbnMsdWlkPW9zLmdldHVpZCgpLGNwdV9hZmZpbml0eT1zb3J0ZWQob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpKQogIGFzc2VydCBwbGF0Zm9ybS5weXRob25fdmVyc2lvbigpPT0nMy4xMy4xNScsJ1B5dGhvbiBwaW4gbWlzbWF0Y2gnCiAgYXNzZXJ0IHZlcnNpb25zPT1FWFBFQ1RFRCwnRXhhY3QgcGFja2FnZSBwaW5zIG1pc21hdGNoJwogIGFzc2VydCBvcy5nZXR1aWQoKT09MTAwMDEsJ0RlcGxveWVkIHJ1bnRpbWUgdWlkIG1pc21hdGNoJwogIGFzc2VydCBsZW4ob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpPT00LCdGb3VyIGFzc2lnbmVkIENQVXMgcmVxdWlyZWQnCiAgZm9yIHAsd2FudGVkIGluIFsoJ21lbW9yeS5tYXgnLHN0cigzMCoxMDI0KiozKSksKCdtZW1vcnkuc3dhcC5tYXgnLCcwJyksKCdjcHVzZXQuY3B1cy5lZmZlY3RpdmUnLCcwLTMnKV06CiAgIGFjdHVhbD0oUGF0aCgnL3N5cy9mcy9jZ3JvdXAnKS9wKS5yZWFkX3RleHQoKS5zdHJpcCgpO3JbcF09YWN0dWFsCiAgIGFzc2VydCBhY3R1YWw9PXdhbnRlZCwnRGVwbG95ZWQgY2dyb3VwIG1pc21hdGNoOiAnK3AKICBhc3NlcnQgbm90IFBhdGgoJy92YXIvcnVuL2RvY2tlci5zb2NrJykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgRG9ja2VyIHNvY2tldCcKICBhc3NlcnQgbm90IFBhdGgoJy9ldGMvc291cC1xdWV1ZS93b3JrZXIuZW52JykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgd29ya2VyIGVudmlyb25tZW50IGZpbGUnCiAgYXNzZXJ0IG5vdCBhbnkoJ1RPS0VOJyBpbiBrIGFuZCBrIT0nVE9LRU5JWkVSU19QQVJBTExFTElTTScgZm9yIGsgaW4gb3MuZW52aXJvbiksJ1Rva2VuLWJlYXJpbmcgZW52aXJvbm1lbnQgcmVqZWN0ZWQgd2l0aG91dCByZWFkaW5nIGl0cyB2YWx1ZScKICBhc3NlcnQgbm90IG9zLmVudmlyb24uZ2V0KCdUUklUT05fSU5URVJQUkVUJyksJ0ludGVycHJldGVyIG1vZGUgZm9yYmlkZGVuJwogIGltcG9ydCB0b3JjaCx0cml0b24sbnVtcHkKICBhc3NlcnQgdG9yY2guX192ZXJzaW9uX189PScyLjExLjArY3UxMjgnIGFuZCB0b3JjaC52ZXJzaW9uLmN1ZGE9PScxMi44JwogIGFzc2VydCB0cml0b24uX192ZXJzaW9uX189PSczLjYuMCcgYW5kIG51bXB5Ll9fdmVyc2lvbl9fPT0nMi4zLjUnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCk9PTEgYW5kIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApPT0nVGVzbGEgVDQnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApPT0oNyw1KQogIHIudXBkYXRlKGdwdV9uYW1lPXRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLGN1ZGFfYnVpbGQ9dG9yY2gudmVyc2lvbi5jdWRhLHRvcmNoX2dpdD10b3JjaC52ZXJzaW9uLmdpdF92ZXJzaW9uKQogIGZyb20gaHVnZ2luZ2ZhY2VfaHViLmNvbnN0YW50cyBpbXBvcnQgSEZfSFVCX0NBQ0hFCiAgclsncmVzb2x2ZWRfcHVibGljX2h1Yl9jYWNoZSddPXN0cihQYXRoKEhGX0hVQl9DQUNIRSkucmVzb2x2ZSgpKQogIHJbJ3Bhc3NlZCddPVRydWUKIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGU6clsnZXJyb3InXT1yZXByKGUpCiBvdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKHIsaW5kZW50PTIpKydcbicpO3ByaW50KGpzb24uZHVtcHMocixpbmRlbnQ9MikpCiByYWlzZSBTeXN0ZW1FeGl0KDAgaWYgclsncGFzc2VkJ10gZWxzZSAxKQppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'prepare_asset_manifest.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJSZWFkL2hhc2ggb25seSB0aGUgc2VsZWN0ZWQgcHVibGljIG1vZGVsIGNhY2hlICsgcHVibGljIHRhc2sgZmlsZXM7IG5vIGRvd25sb2FkLiIiIgppbXBvcnQgYXJncGFyc2UsaGFzaGxpYixqc29uLHJlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApNT0RFTD0nbW9kZWxzLS1Rd2VuLS1Rd2VuMi41LTEuNUItSW5zdHJ1Y3QnClJFVklTSU9OPSc5ODlhYTc5ODBlNGNmODA2ZjgwYzdmZWYyYjFhZGI3YmM3MWFhMzA2JwpEQVRBPXsnZGF0YS90cmFpbi5qc29ubCc6JzkzZDAzNmQ5MTJiZDgxYzMzN2RjMjk4OTNlMTNhNzZmMzYxNjFkOTZmODk2ZWY0ZTAyNDkyZTJmMWEyMWEwZGQnLCdkYXRhL2V2YWxfcHVibGljLmpzb25sJzonYjdmODViNzhiYTMxODZkY2U0ZGVhYjdjZmI4YmY4ZGNmNzk5OGUxNTE3YzU3NDlhYzBmNTAwNTg4NmM2ZjcwZSd9CmRlZiBzaGEocGF0aCk6CiBoPWhhc2hsaWIuc2hhMjU2KCkKIHdpdGggcGF0aC5vcGVuKCdyYicpIGFzIGY6CiAgZm9yIGIgaW4gaXRlcihsYW1iZGE6Zi5yZWFkKDQqMTAyNCoxMDI0KSxiJycpOmgudXBkYXRlKGIpCiByZXR1cm4gaC5oZXhkaWdlc3QoKQpkZWYgYnVpbGQoY2FjaGUscmVwbyxyZXZpc2lvbj1SRVZJU0lPTik6CiBjYWNoZT1QYXRoKGNhY2hlKS5yZXNvbHZlKCk7cmVwbz1QYXRoKHJlcG8pLnJlc29sdmUoKQogaWYgbm90IHJlLmZ1bGxtYXRjaCgnWzAtOWEtZl17NDB9JyxyZXZpc2lvbik6cmFpc2UgVmFsdWVFcnJvcignUmV2aXNpb24gbXVzdCBiZTQwaGV4JykKIHJvb3Q9Y2FjaGUvTU9ERUwKIGlmIHJvb3QuaXNfc3ltbGluaygpIG9yIG5vdCByb290LmlzX2RpcigpOnJhaXNlIFZhbHVlRXJyb3IoJ1NlbGVjdGVkIG1vZGVsIGNhY2hlIG1pc3Npbmcgb3Igc3ltbGlua2VkJykKIGZvciBkaXJlY3RvcnkgaW4gKHJvb3QvJ3JlZnMnLHJvb3QvJ3NuYXBzaG90cycscm9vdC8nYmxvYnMnKToKICBpZiBkaXJlY3RvcnkuaXNfc3ltbGluaygpIG9yIG5vdCBkaXJlY3RvcnkuaXNfZGlyKCk6cmFpc2UgVmFsdWVFcnJvcignTW9kZWwgbWV0YWRhdGEvYmxvYiBkaXJlY3RvcmllcyBtdXN0IGJlIHJlYWwgbG9jYWwgZGlyZWN0b3JpZXMnKQogcmVmPXJvb3QvJ3JlZnMvbWFpbicKIGlmIHJlZi5pc19zeW1saW5rKCkgb3Igbm90IHJlZi5pc19maWxlKCkgb3IgcmVmLnJlYWRfdGV4dCgpLnN0cmlwKCkhPXJldmlzaW9uOnJhaXNlIFZhbHVlRXJyb3IoJ3JlZnMvbWFpbiBtdXN0IGlkZW50aWZ5IGV4YWN0IHBpbm5lZCByZXZpc2lvbjsgbm8gc2lsZW50IHJldmlzaW9uIHN1YnN0aXR1dGlvbicpCiBzbmFwc2hvdD1yb290LydzbmFwc2hvdHMnL3JldmlzaW9uCiBpZiBzbmFwc2hvdC5pc19zeW1saW5rKCkgb3Igbm90IHNuYXBzaG90LmlzX2RpcigpOnJhaXNlIFZhbHVlRXJyb3IoJ1NuYXBzaG90IGRpcmVjdG9yeSBhYnNlbnQ7IHJlc3RvcmUgdmFsaWRhdGVkIHJldGFpbmVkIGxpbmtzIHNlcGFyYXRlbHksIGRvIG5vdCBndWVzcycpCiBmaWxlcz17c3RyKHJlZi5yZWxhdGl2ZV90byhjYWNoZSkpOnNoYShyZWYpfTttZW1vPXt9CiBmb3IgcCBpbiBzb3J0ZWQoc25hcHNob3Qucmdsb2IoJyonKSk6CiAgaWYgcC5pc19kaXIoKToKICAgaWYgcC5pc19zeW1saW5rKCk6cmFpc2UgVmFsdWVFcnJvcignTGlua2VkIHNuYXBzaG90IGRpcmVjdG9yeSByZWplY3RlZCcpCiAgIGNvbnRpbnVlCiAgcmVzb2x2ZWQ9cC5yZXNvbHZlKHN0cmljdD1UcnVlKQogIGlmIG5vdCByZXNvbHZlZC5pc19yZWxhdGl2ZV90bygocm9vdC8nYmxvYnMnKS5yZXNvbHZlKCkpIG9yIG5vdCByZXNvbHZlZC5pc19maWxlKCk6cmFpc2UgVmFsdWVFcnJvcignU25hcHNob3QgdGFyZ2V0IG11c3QgcmVtYWluIGluc2lkZSBzZWxlY3RlZCBtb2RlbCBibG9icycpCiAgaWYgcmVzb2x2ZWQgbm90IGluIG1lbW86bWVtb1tyZXNvbHZlZF09c2hhKHJlc29sdmVkKQogIGZpbGVzW3N0cihwLnJlbGF0aXZlX3RvKGNhY2hlKSldPW1lbW9bcmVzb2x2ZWRdCiBpZiBsZW4oZmlsZXMpPDI6cmFpc2UgVmFsdWVFcnJvcignRW1wdHkgbW9kZWwgc25hcHNob3QnKQogZGF0YT17fQogZm9yIG5hbWUsd2FudGVkIGluIERBVEEuaXRlbXMoKToKICBwPXJlcG8vbmFtZQogIGlmIG5vdCBwLmlzX2ZpbGUoKSBvciBwLmlzX3N5bWxpbmsoKSBvciBub3QgcC5yZXNvbHZlKCkuaXNfcmVsYXRpdmVfdG8ocmVwbykgb3Igc2hhKHApIT13YW50ZWQ6cmFpc2UgVmFsdWVFcnJvcignUHVibGljIGRhdGEgaGFzaCBtaXNtYXRjaDogJytuYW1lKQogIGRhdGFbbmFtZV09d2FudGVkCiByZXR1cm4geydtb2RlbF9pZCc6J1F3ZW4vUXdlbjIuNS0xLjVCLUluc3RydWN0JywncmV2aXNpb24nOnJldmlzaW9uLCdmaWxlcyc6ZmlsZXN9LGRhdGEKaWYgX19uYW1lX189PSdfX21haW5fXyc6CiBwPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pO3AuYWRkX2FyZ3VtZW50KCctLWNhY2hlJyxyZXF1aXJlZD1UcnVlKTtwLmFkZF9hcmd1bWVudCgnLS1yZXBvJyxyZXF1aXJlZD1UcnVlKTtwLmFkZF9hcmd1bWVudCgnLS1vdXQnLHJlcXVpcmVkPVRydWUpO2E9cC5wYXJzZV9hcmdzKCk7bWFuaWZlc3QsZGF0YT1idWlsZChhLmNhY2hlLGEucmVwbyk7b3V0PVBhdGgoYS5vdXQpCiB3aXRoIG91dC5vcGVuKCd4JykgYXMgZjpqc29uLmR1bXAobWFuaWZlc3QsZixpbmRlbnQ9Mik7Zi53cml0ZSgnXG4nKQogcHJpbnQoanNvbi5kdW1wcyh7J21vZGVsX2NhY2hlX21hbmlmZXN0JzpzdHIob3V0KSwnbW9kZWxfY2FjaGVfbWFuaWZlc3Rfc2hhMjU2JzpzaGEob3V0KSwnZGF0YV9maWxlcyc6ZGF0YX0saW5kZW50PTIpKQo=', 'process_guard.py': 'IiIiSXNvbGF0ZWQgTGludXggc3VicmVhcGVyIGd1YXJkaWFuLiBPd25zIGNoaWxkcmVuIGFuZCBob2xkcyB0aGUgR1BVIGxvY2suCgpUaGUgZ3VhcmRpYW4gd2F0Y2hlcyBhIHBhcmVudC1saXZlbmVzcyBwaXBlOiBkcml2ZXIgU0lHS0lMTCBvciBvdGhlciBhYnJ1cHQKZGVhdGggY2xvc2VzIHRoZSBwaXBlIGFuZCB0cmlnZ2VycyBxdWFyYW50aW5lZCBjbGVhbnVwLiBQSURGRCBzaWduYWxpbmcgaW4gdGhlCnNoYXJlZCB0YXNrIGhlbHBlciBwcmV2ZW50cyBQSUQgcmV1c2UgcmFjZXM7IGRlc2NlbmRhbnRzIGFyZSBvbmx5IHRoaXMgZGVkaWNhdGVkCmd1YXJkaWFuJ3MgY2hpbGRyZW4sIGluY2x1ZGluZyBuZXdseSBhZG9wdGVkIG9ycGhhbnMgZnJvbSBuZXN0ZWQgc2Vzc2lvbnMuClRoaXMgaXMgdGFzayBvcmNoZXN0cmF0aW9uLCBuZXZlciBzb2x1dGlvbi8gd29yayBvciBrZXJuZWwgcGFydGljaXBhdGlvbi4KIiIiCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNlbGVjdAppbXBvcnQgc2lnbmFsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKCmltcG9ydCBvd25lZF9wcm9jZXNzX3RyZWUgYXMgdHJlZQoKCmRlZiBxdWFyYW50aW5lKHBhdGgsIHJlYXNvbik6CiAgICAiIiJGaXJzdCBmYWlsdXJlIHdpbnM7IG5ldmVyIG92ZXJ3cml0ZSBvciBhdXRvbWF0aWNhbGx5IGNsZWFyIGEgbWFya2VyLiIiIgogICAgdHJ5OgogICAgICAgIGRlc2NyaXB0b3IgPSBvcy5vcGVuKHBhdGgsIG9zLk9fV1JPTkxZIHwgb3MuT19DUkVBVCB8IG9zLk9fRVhDTCB8IG9zLk9fTk9GT0xMT1csIDBvNjAwKQogICAgZXhjZXB0IEZpbGVFeGlzdHNFcnJvcjoKICAgICAgICByZXR1cm4KICAgIHdpdGggb3MuZmRvcGVuKGRlc2NyaXB0b3IsICJ3IikgYXMgc3RyZWFtOgogICAgICAgIGpzb24uZHVtcCh7InN0YXR1cyI6ICJPUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCIsICJyZWFzb24iOiByZWFzb24sCiAgICAgICAgICAgICAgICAgICAiZ3VhcmRpYW5fcGlkIjogb3MuZ2V0cGlkKCksICJjcmVhdGVkX3VuaXgiOiB0aW1lLnRpbWUoKX0sIHN0cmVhbSkKICAgICAgICBzdHJlYW0ud3JpdGUoIlxuIikKICAgICAgICBzdHJlYW0uZmx1c2goKQogICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkKCgpkZWYgX3dyaXRlX3JlY2VpcHQocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBlbmRpbmcgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAiLnBlbmRpbmciKQogICAgcGVuZGluZy53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUsIGluZGVudD0yKSArICJcbiIpCiAgICBwZW5kaW5nLnJlcGxhY2UocGF0aCkKCgpkZWYgZ3VhcmRpYW5fbWFpbigpOgogICAgY29uZmlnID0ganNvbi5sb2FkcyhzeXMuc3RkaW4ucmVhZGxpbmUoKSkKICAgIHJlYWRfZmQsIGxvY2tfZmQgPSBjb25maWdbInBhcmVudF9waXBlX2ZkIl0sIGNvbmZpZ1sibG9ja19mZCJdCiAgICBzdGFydGVkLCByZXF1ZXN0ZWRfc2lnbmFsLCBwcm9jZXNzID0gdGltZS5tb25vdG9uaWMoKSwgW05vbmVdLCBOb25lCiAgICByZXN1bHQgPSB7ImV4aXRfY29kZSI6IDEyNSwgInJvb3RfZXhpdF9jb2RlIjogTm9uZSwgInRpbWVkX291dCI6IEZhbHNlLAogICAgICAgICAgICAgICJpbnRlcnJ1cHRlZCI6IEZhbHNlLCAiY2xlYW51cF9jb25maXJtZWQiOiBGYWxzZSwKICAgICAgICAgICAgICAicmVhc29uIjogTm9uZSwgImd1YXJkaWFuX3BpZCI6IG9zLmdldHBpZCgpfQogICAgZGVmIHJlcXVlc3Qoc2lnbnVtLCBmcmFtZSk6CiAgICAgICAgcmVxdWVzdGVkX3NpZ25hbFswXSA9IHNpZ251bQogICAgZm9yIHNpZyBpbiAoc2lnbmFsLlNJR0lOVCwgc2lnbmFsLlNJR1RFUk0pOgogICAgICAgIHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KQogICAgdHJ5OgogICAgICAgIHRyZWUucHJlcGFyZSgpICAjIHJlcXVpcmVkIGJlZm9yZSBhbnkgY2hpbGQ7IG5ldmVyIHNpbGVudGx5IGRlZ3JhZGUKICAgICAgICB3aXRoIFBhdGgoY29uZmlnWyJsb2ciXSkub3BlbigieCIpIGFzIHN0cmVhbToKICAgICAgICAgICAgcHJvY2VzcyA9IHN1YnByb2Nlc3MuUG9wZW4oY29uZmlnWyJjb21tYW5kIl0sIGN3ZD1jb25maWdbImN3ZCJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGRvdXQ9c3RyZWFtLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0YXJ0X25ld19zZXNzaW9uPVRydWUpCiAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICByZWFkeSwgXywgXyA9IHNlbGVjdC5zZWxlY3QoW3JlYWRfZmRdLCBbXSwgW10sIDAuMDUpCiAgICAgICAgICAgICAgICBpZiByZWFkeSBhbmQgb3MucmVhZChyZWFkX2ZkLCAxKSA9PSBiIiI6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249InBhcmVudF9saXZlbmVzc19waXBlX2Nsb3NlZCIsIGludGVycnVwdGVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGlmIHJlcXVlc3RlZF9zaWduYWxbMF0gaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX3NpZ25hbF8iICsgc3RyKHJlcXVlc3RlZF9zaWduYWxbMF0pLCBpbnRlcnJ1cHRlZD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZCA+PSBjb25maWdbInRpbWVvdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHQudXBkYXRlKHJlYXNvbj0ib3V0ZXJfdGltZW91dCIsIHRpbWVkX291dD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICByZXR1cm5jb2RlID0gcHJvY2Vzcy5wb2xsKCkKICAgICAgICAgICAgICAgIGlmIHJldHVybmNvZGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyb290X2V4aXRfY29kZSJdID0gcmV0dXJuY29kZQogICAgICAgICAgICAgICAgICAgIGlmIHRyZWUuZGVzY2VuZGFudHMoKToKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyZWFzb24iXSA9ICJyb290X2V4aXRlZF93aXRoX293bmVkX2Rlc2NlbmRhbnRzIgogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUoZXhpdF9jb2RlPXJldHVybmNvZGUsIGNsZWFudXBfY29uZmlybWVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWYgcmVzdWx0WyJyZWFzb24iXSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHF1YXJhbnRpbmUoY29uZmlnWyJtYXJrZXIiXSwgcmVzdWx0WyJyZWFzb24iXSkKICAgICAgICAgICAgICAgICMgY2xlYW51cCBpZ25vcmVzIHJlcGVhdGVkIElOVC9URVJNLCBzdG9wcy9yZXNjYW5zL2tpbGxzIGJ5CiAgICAgICAgICAgICAgICAjIHBpZGZkIGFuZCB3YWl0cy9yZWFwcyB3aXRob3V0IGFueSB0aW1lb3V0LXVubG9jayBlc2NhcGUuCiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUocm9vdF9leGl0X2NvZGU9cHJvY2Vzcy5yZXR1cm5jb2RlLCBjbGVhbnVwX2NvbmZpcm1lZD1UcnVlKQogICAgICAgICAgICAgICAgaWYgcmVzdWx0WyJ0aW1lZF9vdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTI0CiAgICAgICAgICAgICAgICBlbGlmIHJlc3VsdFsiaW50ZXJydXB0ZWQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTMwCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX2V4Y2VwdGlvbjoiICsgcmVwcihleGMpLCBleGl0X2NvZGU9MTI1KQogICAgICAgIHRyeToKICAgICAgICAgICAgcXVhcmFudGluZShjb25maWdbIm1hcmtlciJdLCByZXN1bHRbInJlYXNvbiJdKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGlmIHByb2Nlc3MgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgcmVzdWx0WyJjbGVhbnVwX2NvbmZpcm1lZCJdID0gbm90IHRyZWUuZGVzY2VuZGFudHMoKQogICAgZmluYWxseToKICAgICAgICByZXN1bHRbIm91dGVyX3dhbGxfcyJdID0gdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWQKICAgICAgICBfd3JpdGVfcmVjZWlwdChjb25maWdbInJlY2VpcHQiXSwgcmVzdWx0KQogICAgICAgIG9zLmNsb3NlKHJlYWRfZmQpCiAgICAgICAgIyBUaGUgaW5oZXJpdGVkIG9wZW4tZmlsZSBkZXNjcmlwdGlvbiByZXRhaW5zIHRoZSBjYW5vbmljYWwgZmxvY2sgZXZlbgogICAgICAgICMgaWYgdGhlIGRyaXZlciBoYXMgZGllZC4gQ2xvc2UgT05MWSBhZnRlciBvd25lZC10cmVlIGNsZWFudXAgY29tcGxldGVzLgogICAgICAgIG9zLmNsb3NlKGxvY2tfZmQpCiAgICByZXR1cm4gcmVzdWx0WyJleGl0X2NvZGUiXQoKCmRlZiBydW4oY29tbWFuZCwgKiwgY3dkLCBlbnYsIGxvZywgdGltZW91dCwgbG9ja19mZCwKICAgICAgICBtYXJrZXI9Ii90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCIpOgogICAgIiIiRGVkaWNhdGVkIGRyaXZlciBBUEkuIEFueSBCYXNlRXhjZXB0aW9uIGFza3MgZ3VhcmRpYW4gY2xlYW51cCBhbmQgd2FpdHMuIiIiCiAgICByZWNlaXB0ID0gUGF0aChzdHIobG9nKSArICIucHJvY2Vzcy5qc29uIikucmVzb2x2ZSgpCiAgICBpZiByZWNlaXB0LmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUHJvY2VzcyByZWNlaXB0IGFscmVhZHkgZXhpc3RzOyByZWZ1c2UgdG8gb3ZlcndyaXRlIikKICAgIHJlYWRfZmQsIHdyaXRlX2ZkID0gb3MucGlwZSgpCiAgICBjb25maWcgPSB7ImNvbW1hbmQiOiBbc3RyKHgpIGZvciB4IGluIGNvbW1hbmRdLCAiY3dkIjogc3RyKFBhdGgoY3dkKS5yZXNvbHZlKCkpLAogICAgICAgICAgICAgICJsb2ciOiBzdHIoUGF0aChsb2cpLnJlc29sdmUoKSksICJ0aW1lb3V0IjogdGltZW91dCwKICAgICAgICAgICAgICAicmVjZWlwdCI6IHN0cihyZWNlaXB0KSwgInBhcmVudF9waXBlX2ZkIjogcmVhZF9mZCwKICAgICAgICAgICAgICAibG9ja19mZCI6IGxvY2tfZmQsICJtYXJrZXIiOiBzdHIobWFya2VyKX0KICAgIGd1YXJkaWFuID0gTm9uZQogICAgaW50ZXJydXB0ZWQgPSBOb25lCiAgICBkZWYgcmVxdWVzdChzaWdudW0sIGZyYW1lKToKICAgICAgICByYWlzZSBJbnRlcnJ1cHRlZEVycm9yKCJEcml2ZXIgcmVjZWl2ZWQgc2lnbmFsICIgKyBzdHIoc2lnbnVtKSkKICAgIG9sZF9oYW5kbGVycyA9IHtzaWc6IHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KSBmb3Igc2lnIGluIChzaWduYWwuU0lHSU5ULCBzaWduYWwuU0lHVEVSTSl9CiAgICB0cnk6CiAgICAgICAgZ3VhcmRpYW4gPSBzdWJwcm9jZXNzLlBvcGVuKFtzeXMuZXhlY3V0YWJsZSwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKSksICItLWd1YXJkaWFuIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZGluPXN1YnByb2Nlc3MuUElQRSwgZW52PWVudiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFzc19mZHM9KHJlYWRfZmQsIGxvY2tfZmQpLCBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0ZXh0PVRydWUpCiAgICAgICAgb3MuY2xvc2UocmVhZF9mZCkKICAgICAgICByZWFkX2ZkID0gTm9uZQogICAgICAgIGd1YXJkaWFuLnN0ZGluLndyaXRlKGpzb24uZHVtcHMoY29uZmlnKSArICJcbiIpCiAgICAgICAgZ3VhcmRpYW4uc3RkaW4uY2xvc2UoKQogICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIGludGVycnVwdGVkID0gcmVwcihleGMpCiAgICAgICAgIyBJZ25vcmUgcmVwZWF0ZWQgc3RvcCBzaWduYWxzIHVudGlsIHRoZSBndWFyZGlhbiBjb25maXJtcyBjbGVhbnVwLgogICAgICAgIGZvciBzaWcgaW4gb2xkX2hhbmRsZXJzOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgc2lnbmFsLlNJR19JR04pCiAgICAgICAgaWYgd3JpdGVfZmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG9zLmNsb3NlKHdyaXRlX2ZkKQogICAgICAgICAgICB3cml0ZV9mZCA9IE5vbmUKICAgICAgICBpZiBndWFyZGlhbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgaWYgZ3VhcmRpYW4uc3RkaW4gaXMgbm90IE5vbmUgYW5kIG5vdCBndWFyZGlhbi5zdGRpbi5jbG9zZWQ6CiAgICAgICAgICAgICAgICBndWFyZGlhbi5zdGRpbi5jbG9zZSgpCiAgICAgICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZmluYWxseToKICAgICAgICBpZiByZWFkX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZShyZWFkX2ZkKQogICAgICAgIGlmIHdyaXRlX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZSh3cml0ZV9mZCkKICAgICAgICBmb3Igc2lnLCBoYW5kbGVyIGluIG9sZF9oYW5kbGVycy5pdGVtcygpOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgaGFuZGxlcikKICAgIGlmIG5vdCByZWNlaXB0LmlzX2ZpbGUoKToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX21pc3NpbmdfcmVjZWlwdCIpCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJHdWFyZGlhbiByZWNlaXB0IG1pc3Npbmc7IEdQVSBxdWFyYW50aW5lIHJlcXVpcmVzIG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uIikKICAgIHJlc3VsdCA9IGpzb24ubG9hZHMocmVjZWlwdC5yZWFkX3RleHQoKSkKICAgIGlmIGludGVycnVwdGVkIGlzIG5vdCBOb25lOgogICAgICAgIHJlc3VsdC51cGRhdGUocGFyZW50X2V4Y2VwdGlvbj1pbnRlcnJ1cHRlZCwgaW50ZXJydXB0ZWQ9VHJ1ZSkKICAgICAgICBpZiByZXN1bHRbImV4aXRfY29kZSJdID09IDA6CiAgICAgICAgICAgIHJlc3VsdFsiZXhpdF9jb2RlIl0gPSAxMzAKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImRyaXZlcl9pbnRlcnJ1cHRpb246IiArIGludGVycnVwdGVkKQogICAgaWYgbm90IHJlc3VsdFsiY2xlYW51cF9jb25maXJtZWQiXToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX3VuY2VydGFpbl9jbGVhbnVwIikKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkd1YXJkaWFuIGNsZWFudXAgdW5jZXJ0YWluOyBxdWFyYW50aW5lIHJldGFpbmVkIikKICAgIHJldHVybiByZXN1bHQKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaWYgc3lzLmFyZ3ZbMTpdICE9IFsiLS1ndWFyZGlhbiJdOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIkludGVybmFsIGd1YXJkaWFuIGVudHJ5cG9pbnQgb25seSIpCiAgICByYWlzZSBTeXN0ZW1FeGl0KGd1YXJkaWFuX21haW4oKSkK', 'evidence_reader.py': 'IiIiUHJvcG9zZWQgbmV4dC12ZXJzaW9uIGxvZyByZWFkZXI7IGRvZXMgbm90IGFsdGVyIGFueSBydW4vY29tcGxldGlvbiBkZWNpc2lvbi4KCkFjY2VwdCBhIHN0cnVjdHVyZWQgSlNPTiBvYmplY3Qgb25seSBhZnRlciBhbiBleGFjdCBsaW5lLWFuY2hvcmVkIGV2ZW50IHByZWZpeCwKaWdub3JpbmcgdGVybWluYWwgQU5TSSBlc2NhcGUgZm9ybWF0dGluZyBhbmQgSlNPTiB3aGl0ZXNwYWNlIGxpbmUgd3JhcHBpbmcuCk1hbGZvcm1lZC90cnVuY2F0ZWQvb3ZlcnNpemUvZHVwbGljYXRlLWtleS9ub25maW5pdGUgb2JqZWN0cyBmYWlsIGNsb3NlZC4KIiIiCmltcG9ydCBqc29uCmltcG9ydCByZQoKIyBDU0ksIE9TQyAoQkVMIG9yIFNUIHRlcm1pbmF0aW9uKSwgYW5kIHN0YW5kYXJkIHR3by1jaGFyYWN0ZXIgQU5TSSBlc2NhcGVzLgpfQU5TSSA9IHJlLmNvbXBpbGUocidceDFiXF1bXlx4MDdceDFiXSooPzpceDA3fFx4MWJcXCl8XHgxYlxbWzAtP10qWyAtL10qW0Atfl18XHgxYltALV9dJykKCmRlZiBfcmVqZWN0KHZhbHVlKToKICAgIHJhaXNlIFZhbHVlRXJyb3IoJ05vbmZpbml0ZSBKU09OIGNvbnN0YW50OiAnICsgdmFsdWUpCgoKZGVmIF91bmlxdWVfcGFpcnMocGFpcnMpOgogICAgcmVzdWx0ID0ge30KICAgIGZvciBrZXksIHZhbHVlIGluIHBhaXJzOgogICAgICAgIGlmIGtleSBpbiByZXN1bHQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0R1cGxpY2F0ZSBKU09OIGtleTogJyArIGtleSkKICAgICAgICByZXN1bHRba2V5XSA9IHZhbHVlCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIGV2ZW50cyh0ZXh0LCBwcmVmaXgsICosIG1heF9ldmVudF9jaGFycz0zMjc2OCk6CiAgICAiIiJZaWVsZCB1bmFtYmlndW91cyBkaWN0IGV2ZW50czsgbmV2ZXIgaW5mZXIgZnVsbC1ydW4gb3IgQ1VEQSBjb21wbGV0aW9uLiIiIgogICAgaWYgbm90IGlzaW5zdGFuY2UocHJlZml4LCBzdHIpIG9yIG5vdCBwcmVmaXggb3IgJ1xuJyBpbiBwcmVmaXggb3IgJ1xyJyBpbiBwcmVmaXg6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQSBub25lbXB0eSBzaW5nbGUtbGluZSBleGFjdCBwcmVmaXggaXMgcmVxdWlyZWQnKQogICAgY2xlYW4gPSBfQU5TSS5zdWIoJycsIHRleHQpLnJlcGxhY2UoJ1xyXG4nLCAnXG4nKS5yZXBsYWNlKCdccicsICdcbicpCiAgICBkZWNvZGVyID0ganNvbi5KU09ORGVjb2RlcihvYmplY3RfcGFpcnNfaG9vaz1fdW5pcXVlX3BhaXJzLCBwYXJzZV9jb25zdGFudD1fcmVqZWN0KQogICAgZm9yIG1hdGNoIGluIHJlLmZpbmRpdGVyKHInKD9tKV5bIFx0XSonICsgcmUuZXNjYXBlKHByZWZpeCksIGNsZWFuKToKICAgICAgICBmcmFnbWVudCA9IGNsZWFuW21hdGNoLmVuZCgpOm1hdGNoLmVuZCgpICsgbWF4X2V2ZW50X2NoYXJzXS5sc3RyaXAoKQogICAgICAgIHRyeToKICAgICAgICAgICAgcmVjb3JkLCBlbmQgPSBkZWNvZGVyLnJhd19kZWNvZGUoZnJhZ21lbnQpCiAgICAgICAgZXhjZXB0IChWYWx1ZUVycm9yLCBUeXBlRXJyb3IsIFJlY3Vyc2lvbkVycm9yKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShyZWNvcmQsIGRpY3QpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIGZyYWdtZW50W2VuZDpdLnNwbGl0KCdcbicsIDEpWzBdLnN0cmlwKCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgeWllbGQgcmVjb3JkCg==', 'official_manifest.json': 'ewogICJoYXJuZXNzL190cmFpbi5weSI6ICI0NWE2MjY0MmRjZWFiMzJjZjNlZmNkYTUwNGRlODdhOWMyNWVhMjhiMTE5YjE4MDZmNTZkMTRmZjExNDZhODlkIiwKICAiaGFybmVzcy9jaGVja19ydWxlcy5weSI6ICI4NzlhYjE4NjE2NzRmNmU3MTRlYWRkMDBhYWYwNDQyMzVkOTg1MjZkZDc1ZmFiMTAzNWRlZWMzNDAxMTIxZWY1IiwKICAiaGFybmVzcy9jb21tb24ucHkiOiAiYzgzMDY2ZjI0ZjI2NmRhYjYyZWJjYzljMmQ0NGY1YTY1OTAyNzYwYjUzZGE5MTJlNjVlYmZlNTExNTc5YzJkYiIsCiAgImhhcm5lc3MvZXZhbF9hZGFwdGVyLnB5IjogIjQ3Y2RkYmI4ZTQ1NzljNjlmOTUzNWIwOTlmZjIzNzFiMWMzYWYwOGJmYmQ5M2QxNGJmNzlhYWRkNzRkMzg0ODIiLAogICJoYXJuZXNzL2V4YW1wbGVfcmVmZXJlbmNlLnB5IjogIjZhMTEwNTVmNGExODIyZTUzOTcxYzIyOGM2ZTNlM2FhNmI4OGVkZDllYjZkYjgxYTM4OGUyNmE3MGEwMzBjZWIiLAogICJoYXJuZXNzL3J1bl90YXNrLnB5IjogImQ5MzVhODk1OWYzNDg5NzczMTNmMmVlZjM2ODcwZmYyNjk5YjliYTAxM2E3OTc4OTE4ZmYzNWUxNDg5NjhhNzkiLAogICJoYXJuZXNzL3Njb3JlLnB5IjogIjczMWM5YjY0NGRjYzQwNjliMGIwNjk2MTZmNDM4MDY0ZDFmN2YzNzZhMjAyZmViMTUyMDFmZDkxMGVjNjg4YzUiLAogICJ0YXNrL3ByZXBhcmVfZGF0YS5weSI6ICJhODU0M2YxMWE1NWFmZjgwZGJkYzcxNGEwZTYyMzkxMjQ0YTUyOTY2MTZlZWEyZTM0ZGMyZWIzZmYzZWMxOGY3IiwKICAidGFzay90YXNrLnlhbWwiOiAiZGQ3MGM3MzlmY2QxNDNlMDZhZWYzYmIxZWZjYzkzODZmNjgyYmYxZGUyNmYzMGUwODQwOWQ4NmE5N2ZmNDdjZSIsCiAgInRlc3RzL2NvbmZ0ZXN0LnB5IjogImE4MmVhYmFiNDY4ZWEwZWYxZTAwM2NjY2Q3YzM4MjdjZTJmNGQwMGQwOWMzMjk0OGEzNGZiNDgyMjFmNzMzN2EiLAogICJ0ZXN0cy90ZXN0X2NhbGxiYWNrLnB5IjogIjE5MTkzMDNhMzY3MGMwYTMzYWJiZmNlOWIzM2VhYjA5MTNiYjY0YTNhMjFjMzMxMmQ4NjUzMWMzMmI1YmY0OTAiLAogICJ0ZXN0cy90ZXN0X2NvbmZpZy5weSI6ICJkYjgwOGM0NDQ3MGQxNDRiN2NmMThiNzMyOTQ5MmY0MWY3NmUzN2NmYzI0MzA2ZTdjYmEwNDg4YmNhOWQ3ODdjIiwKICAidGVzdHMvdGVzdF9mb3JtYXRzLnB5IjogIjVkNjU2ZWQ1MGM1OWY3OTRlOTM0NDczZDQ1OWVhODU2MzVmODkyYmEwMjhlMTkyNDE0NDc0ZTliNWQ2NTkxOWUiLAogICJ0ZXN0cy90ZXN0X2xvYWRlci5weSI6ICIzZGFkMWJlYTJkMzNhYWI5N2U1MmI3Y2Q0NDg4OGQ3NjhiYzk1MTlhZjQ0ZmJlNmNmM2U5NmMyNjAwMjY2MzJmIiwKICAidGVzdHMvdGVzdF90cmFpbmVyX2luaXQucHkiOiAiYjc2MDhkNjJhNDA5ODNiZjdjOWU1YjI3MGUwZTUyN2I2NDFhODM3ZTQ2MmQ2ZGYyOTA5ZmVjODhmNjA3ZTgyMSIKfQo=', 'web_runner.py': '...', 'candidate_profile.json': 'ewogICJmdWxsX21fY29tcGFjdF9zdG9yYWdlX3NpbmdsZV92MSI6IHsKICAgICJtb2RlIjogImZ1bGxfbV9jb21wYWN0X3N0b3JhZ2Vfc2luZ2xlX3YxIiwKICAgICJ0aXRsZSI6ICJFeHBsb3JhdG9yeSBzaW5nbGUgY2FuZGlkYXRlOiBjb21wYWN0IE1MUCArIGF0dGVzdGVkIGxvc3NsZXNzIGZyb3plbiBzdG9yYWdlIiwKICAgICJwcmVmbGlnaHQiOiAic29sdXRpb24uZXhwbG9yYXRvcnlfcHJlZmxpZ2h0IiwKICAgICJwcmVmbGlnaHRfc3RhdHVzIjogIlNUUlVDVFVSQUxfQ0hFQ0tTX0NPTVBMRVRFX05VTUVSSUNBTF9SRVNVTFRTX1JFQ09SREVEIiwKICAgICJrZXJuZWxzIjogWwogICAgICAic291cF9jZV9leHBfdGlsZV9tYXgiLAogICAgICAic291cF9jZV9leHBfcm93X21heCIsCiAgICAgICJzb3VwX2NlX2V4cF90aWxlX3N1bSIsCiAgICAgICJzb3VwX2NlX2V4cF9zZWxlY3RlZCIsCiAgICAgICJzb3VwX2NlX21ldHJpY190aWxlcyIsCiAgICAgICJzb3VwX2NlX21ldHJpY19tZXJnZSIsCiAgICAgICJzb3VwX25mNF9kZXF1YW50aXplIiwKICAgICAgInNvdXBfcm1zbm9ybV9mb3J3YXJkIiwKICAgICAgInNvdXBfcm1zbm9ybV9iYWNrd2FyZCIKICAgIF0sCiAgICAicm91dGVzIjogWwogICAgICB7CiAgICAgICAgInByZWZpeCI6ICJTT1VQX1NUUkVBTUVEX0NFICIsCiAgICAgICAgImV2ZW50IjogIm93bmVkX2NvbnN1bWVyX2d1YXJkIiwKICAgICAgICAicG9zaXRpdmVfZmllbGQiOiAiY291bnQiLAogICAgICAgICJtYXRjaCI6IHsKICAgICAgICAgICJjb3VudCI6IDY0CiAgICAgICAgfQogICAgICB9LAogICAgICB7CiAgICAgICAgInByZWZpeCI6ICJTT1VQX1NUUkVBTUVEX0NFICIsCiAgICAgICAgImV2ZW50IjogImJhY2tib25lX29uY2UiLAogICAgICAgICJwb3NpdGl2ZV9maWVsZCI6ICJjb3VudCIsCiAgICAgICAgIm1hdGNoIjogewogICAgICAgICAgImNvdW50IjogNjQKICAgICAgICB9CiAgICAgIH0sCiAgICAgIHsKICAgICAgICAicHJlZml4IjogIlNPVVBfU1RSRUFNRURfQ0UgIiwKICAgICAgICAiZXZlbnQiOiAic3VtbWFyeV9sb3NzX2NvbXBsZXRlZCIsCiAgICAgICAgInBvc2l0aXZlX2ZpZWxkIjogImNvdW50IiwKICAgICAgICAibWF0Y2giOiB7CiAgICAgICAgICAiY291bnQiOiA2NAogICAgICAgIH0KICAgICAgfSwKICAgICAgewogICAgICAgICJwcmVmaXgiOiAiU09VUF9TVFJFQU1FRF9DRSAiLAogICAgICAgICJldmVudCI6ICJzdHJlYW1pbmdfbG9zc19yZXR1cm4iLAogICAgICAgICJwb3NpdGl2ZV9maWVsZCI6ICJjb3VudCIsCiAgICAgICAgIm1hdGNoIjogewogICAgICAgICAgImNvdW50IjogNjQKICAgICAgICB9CiAgICAgIH0sCiAgICAgIHsKICAgICAgICAicHJlZml4IjogIlNPVVBfU1RSRUFNRURfQ0UgIiwKICAgICAgICAiZXZlbnQiOiAiY29uc3VtZXJfY29tcGxldGUiLAogICAgICAgICJwb3NpdGl2ZV9maWVsZCI6ICJjb3VudCIsCiAgICAgICAgIm1hdGNoIjogewogICAgICAgICAgImNvdW50IjogNjQKICAgICAgICB9CiAgICAgIH0sCiAgICAgIHsKICAgICAgICAicHJlZml4IjogIlNPVVBfQ09NUEFDVF9NTFAgIiwKICAgICAgICAiZXZlbnQiOiAib3duZWRfbWxwX2ZvcndhcmQiLAogICAgICAgICJwb3NpdGl2ZV9maWVsZCI6ICJjb3VudCIsCiAgICAgICAgIm1hdGNoIjogewogICAgICAgICAgImNvdW50IjogMTc5MiwKICAgICAgICAgICJuYXRpdmVfZmFsbGJhY2siOiAwLAogICAgICAgICAgInN0YXJ0ZXJfb3JpZ2luYWxfY3JlZGl0IjogMAogICAgICAgIH0KICAgICAgfSwKICAgICAgewogICAgICAgICJwcmVmaXgiOiAiU09VUF9GUk9aRU5fU1RPUkFHRSAiLAogICAgICAgICJldmVudCI6ICJvd25lZF9lbWJlZGRpbmdfZm9yd2FyZCIsCiAgICAgICAgInBvc2l0aXZlX2ZpZWxkIjogImNvdW50IiwKICAgICAgICAibWF0Y2giOiB7CiAgICAgICAgICAiY291bnQiOiA2NCwKICAgICAgICAgICJpbnN0YWxsZWQiOiAxLAogICAgICAgICAgImhlYWRfZnAxNl9jYWxscyI6IDAsCiAgICAgICAgICAiaGVhZF9uYXRpdmVfb3BlcmFuZF9jYWxscyI6IDAsCiAgICAgICAgICAib3duZXJzX2ludGFjdCI6IHRydWUsCiAgICAgICAgICAic3RvcmFnZV9vcmlnaW5hbF9jcmVkaXQiOiAwCiAgICAgICAgfQogICAgICB9CiAgICBdLAogICAgInJlcXVpcmVzX29yaWdpbmFsX2tlcm5lbHMiOiB0cnVlLAogICAgInppcF9zaGEyNTYiOiAiNGZiODNlNDk4NGM2ZWEyMzBjYzI2ZjA3NTYzMzE1YzcxNjY3YTY4ZGM0MzFhZDJjZGRhODk0NTllMDg4YTkwNSIsCiAgICAidmFsaWRhdGlvbl9wb2xpY3kiOiAiRVhQTElDSVRfRklOSVRFX0xPUkFfR1JBRElFTlRfVVBEQVRFX1dBUk5JTkdTX09OTFkiLAogICAgInN0YXJ0ZXJfaW5jbHVkZWQiOiB0cnVlLAogICAgInN0YXJ0ZXJfb3JpZ2luYWxfY3JlZGl0IjogMCwKICAgICJjb21wYWN0X21scF9tZWNoYW5pc21fYWxyZWFkeV9pbl91c2VyX3BhY2thZ2UiOiB0cnVlLAogICAgImVtYmVkZGluZ19zdG9yYWdlX2NoYW5nZWQiOiB0cnVlLAogICAgImV4Y2x1ZGVkX2tlcm5lbHMiOiBbCiAgICAgICJleGFtcGxlX3N3aWdsdV9md2Rfa2VybmVsIiwKICAgICAgImV4YW1wbGVfc3dpZ2x1X2J3ZF9rZXJuZWwiCiAgICBdLAogICAgImVtYmVkZGluZ19vcmlnaW5hbF9jcmVkaXQiOiAwLAogICAgIm5hdGl2ZV9lbWJlZGRpbmdfbG9va3VwIjogdHJ1ZSwKICAgICJzdG9yYWdlX21lY2hhbmlzbV9wcmVjZWRlbnQiOiB0cnVlCiAgfQp9Cg==', 'full_m_compact_storage_single_v1.zip': '...', 'full_m_compact_storage_single_v1_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICI0MDI2NThmZTQ2OGNiOGJhMGI2Mjk4Yzg4NWUzYmNmMzI0YWY3YzY0MGQ4ZDA5NDE0NDQ0ZGIxZDlkMThmYjdkIiwKICAic29sdXRpb24vY2VfdGlsZS5weSI6ICIyYmFlYTFmNGI3YzU2NzM3OWUzNjc0Yjc2MjA0ZDM5ZTlmYjEyOGViNmNmY2RhOWE4ZDhlZGViYmZkMjMwZTdiIiwKICAic29sdXRpb24vY2hhaW4ucHkiOiAiOTlmMGIxOGNjOThkYTA5NzFmYmJkZWRlMGI3OTZiOThiZmU1NDcyYWE5M2U2MWM5ZGUyOWIwOWViNTI5ZGMyZSIsCiAgInNvbHV0aW9uL2NvbW1vbi9fX2luaXRfXy5weSI6ICJlM2IwYzQ0Mjk4ZmMxYzE0OWFmYmY0Yzg5OTZmYjkyNDI3YWU0MWU0NjQ5YjkzNGNhNDk1OTkxYjc4NTJiODU1IiwKICAic29sdXRpb24vY29tbW9uL2JyaWRnZV9zdGF0ZS5weSI6ICJlM2JhNjQ0NDZhOGJiM2ViOGNkNmVjMzFiNGU1ZDAwMThhNDdjOGYzYjkwZDAwZGE5NTVkYzJlNDQ0ZWIxOGU5IiwKICAic29sdXRpb24vY29tbW9uL2NvbnN1bWVyLnB5IjogImYwMThhOWI2ODA4ZDc2MzgxYTIwNTU4MGU4YTJmZjlkMmMwZWI3MDZmYTM0Yzg0MzRmZTQxYjBhZDk5ZGEyYmMiLAogICJzb2x1dGlvbi9jb21wYWN0X21scF9pbnN0YWxsLnB5IjogIjRjNzk2MDY0OWRhYWY2OTBjMjNiZWI3NDdmODUxNzNjY2Q1YjI0NThmMDY5YzFiMzY2MGYwYzE1NWM1NDczZDQiLAogICJzb2x1dGlvbi9jb250cmFjdHMucHkiOiAiMzZiYjU0YzZjYWM0MGI3ZjM4OTZiMmNlM2E1MjY2NWY5NzI5NDJmNWU1ODU2ZTIzNzk0NmExZjgwMDkxZmI0YiIsCiAgInNvbHV0aW9uL2V4cGxvcmF0b3J5X3ByZWZsaWdodC5weSI6ICI3NTIzYTY5YjAyNDI2N2ZjMGNiNDU3YjRhNDcyYjJhYTNmNTY3MGVlYjc1NWFjMWE5NmRkMDE3OTBlNmQ5ZTIzIiwKICAic29sdXRpb24vZnJvemVuX3N0b3JhZ2UucHkiOiAiYTNlZGZkZmU1Njg1OTVlMTgzY2U1N2Q4NThkYjQwMzJkMmUwZTBiNmNmZTViYWM4NDY4ZGRhM2IwYzI5OTMzOCIsCiAgInNvbHV0aW9uL21scF9oYWxmL19faW5pdF9fLnB5IjogImM4ZTI1YzgzZGE3NzY0OWM3ZTc2ZDdlMjU1N2RjYThhM2I5ZTk3NWE4MWZhNzM4NzEzMjExYTlhOGY4ZDZmZjAiLAogICJzb2x1dGlvbi9tbHBfaGFsZi9uYXRpdmVfb3duZXIucHkiOiAiN2I2OWYyYTM1NGRlZjI1Y2I1M2Q0Y2E3OGYzYzE5MDgyNDk3ODM4YzVmNmVlMTMzZDk0Njk0ZTYxMTNkNTQwYyIsCiAgInNvbHV0aW9uL21scF9oYWxmL293bmVkX2hhbGYucHkiOiAiOGE3MTQyMDVjNGYxZGU4ZmY3OTk3Zjk0MzBlNDdmYjQxN2I1NGU2Njg3OWI2NGZhODAxNDQ4OTZjNTJhODA3YiIsCiAgInNvbHV0aW9uL21scF9oYWxmL3N0YXJ0ZXJfaGFsZi5weSI6ICIzNzU5ZjM3NjFiM2IzZDJhOGUwMThiODBhODU3MWM5Y2MxNDU2YjhhZGMxZWQ0MGU0N2UxYzkxOTEzMmU3ZGRhIiwKICAic29sdXRpb24vbWxwX2hhbGYvc3RhcnRlcl9zd2lnbHUucHkiOiAiZTY2NTkxMGZhZWNhNjA5MWM5NThlZDdhMTkxNjA0NDg0YTQ0MjE2ZjUxZDZmYTA2OWY1ZjQ4MjIwZTc0NzJlNiIsCiAgInNvbHV0aW9uL29yaWdpbmFsL19faW5pdF9fLnB5IjogImUzYjBjNDQyOThmYzFjMTQ5YWZiZjRjODk5NmZiOTI0MjdhZTQxZTQ2NDliOTM0Y2E0OTU5OTFiNzg1MmI4NTUiLAogICJzb2x1dGlvbi9vcmlnaW5hbC9tYXRlcmlhbGl6ZWRfY2UucHkiOiAiNjUxOWM4ZjFkZGNlN2RiMjcwZmJlNDI1YThlODc3MWFlY2RhM2YyOTRlY2M4MzgzOWQ0NzU5NTMxZmQxZGNlMSIsCiAgInNvbHV0aW9uL29yaWdpbmFsL21ldHJpY19zdW1tYXJ5LnB5IjogIjhjNWVkZWEwMGM3ZjBiNDI1MDFjNzFlMzRhOTg2ZGUwNWMyOGMzYzU2ZWRmNGQ2MTgxZDJkZGFmMjQxNDJjZjQiLAogICJzb2x1dGlvbi9vcmlnaW5hbC9taXhlZF9yb3dzLnB5IjogIjA5MTE1N2NkYjkwNGIwYTJmZWUwOWQ3MjA4NjJiMzcwZDhmNzJmOWQ4MjQwMjNkMzdlZGJkZjljNWUxNzI2YjIiLAogICJzb2x1dGlvbi9vd25lcnNoaXAucHkiOiAiNzg0YzU2ZGQ1YzBjOGY2ZGNhMTMxMWIzOGE3MWYwN2Q3ODA5ODI0YWI4Y2UzMzQ1NjU3OTAwNWIxZTc3NmYwNSIsCiAgInNvbHV0aW9uL3NvdXJjZV9vd25lcnMucHkiOiAiYzcwMWJmZDcwNjNlMjg5ZDkxYzhlNWI4NTZiNjI4MjdmNmE5ZjRmMjU3OWFhMjNiYmVkNGE5MTFhNWQzOGU1YiIsCiAgInNvbHV0aW9uL3N0cmVhbWluZ19ib3VuZGFyeS5weSI6ICI3NWZlOGJmOTU4MjEzYTM3ODJlYWZhMDIzZTRhM2QxY2Q0N2NkY2E3NDgzNTU2MmJkODFhZDc4MjdhOTY0OGFhIiwKICAic29sdXRpb24vc3VwcG9ydC9fX2luaXRfXy5weSI6ICJlM2IwYzQ0Mjk4ZmMxYzE0OWFmYmY0Yzg5OTZmYjkyNDI3YWU0MWU0NjQ5YjkzNGNhNDk1OTkxYjc4NTJiODU1IiwKICAic29sdXRpb24vc3VwcG9ydC9uZjQucHkiOiAiOWUwN2E3M2Q0NWY3YjcxMTlmYzFmNWQxZDM0MjIzNTFlNWQzNGUyZmJkOTYxMmY3YjkwZjU2YmMxN2FkNjk3YSIsCiAgInNvbHV0aW9uL3N1cHBvcnQvcm1zbm9ybS5weSI6ICJhYTU1YTIxZmQ5MjFkMmY1NDc4NDQ0YjAzYTMyNjk0YTMyOWJiMTBiYmJlNGYzYmY0NmU0ZTc4ZjI4MDRhOTNjIiwKICAic29sdXRpb24vc3VwcG9ydF9pbnN0YWxsLnB5IjogImNjMmI0MDA1OTdiYjM1MjQ2YzM1Mzg5MzE4NWMzNzA4NzM5NGJiNmMwZTVlOWEzYTY3N2YyMWRjMmJmMzg5ODQiLAogICJzb2x1dGlvbi90ZWxlbWV0cnkucHkiOiAiODZmYmY3M2RmMDdjOTE3OTQ3NGE5NGYyYzUzNDlmMDRkOGEzNmRmMTZlZDNlMDg0ZWM5NDMzY2IzMDU1NjYyYyIsCiAgInNvbHV0aW9uL3Rva2VuX2NodW5rX2hlYWQucHkiOiAiYmNjNjQxN2U0ZDM3NjkyMDY2OGJjYjM0MWY1NDkxYWZlMzI4NDdkNzY4YTdkMTBkZjc1NTAzYjFjOWUyNDRhYyIsCiAgInNvbHV0aW9uL3ZlcmlmeV90aWxlLnB5IjogImY0ZjYyNTE5ODdiNjE5OTM2NWI4NTNkNDg5NzlhMTM2ZWMyZGMyNjI2YmVkNjhmMzc0YWUyOTQwZWU2MDQ3YTEiLAogICJzb2x1dGlvbi93ZWJfcHJlZmxpZ2h0LnB5IjogImZiMmFlOTIyOGI0NjlhZTZkNDc2MDkyM2FiNTdmNTc1ZjVkY2RmYWYzODNhYTNlODVmZjhkMzcxMGQzM2M1ZTEiCn0K'}
PAYLOAD_SHA256 = {'README.md': '15b1b23c69df23f8048d2925c991c2d9bad8df7159664c3d6eb9cd09f17d216d', 'runtime_api_reference.json': 'd79e555702154a0516efa87ca22ab00dad39c515eefb12fc7332a0da40108881', 'owned_process_tree.py': '693dde708da5c06053af239463952115ef5e8865f50bcbc51a2038237ca57d69', 'runtime_probe.py': 'efe1b7cf3f3cf1e37bbc169a8468859c5127ec3281a37e39068703b2eb3c1480', 'prepare_asset_manifest.py': 'f2a0f36b5cd367d506c831c69f1a39d49c38c3c815758be5d0fd7a2ec033f46d', 'process_guard.py': '82465943a3a9af3831a5399a3a55d09ac290ba7365c4e80d6bc5e79fab62a6ff', 'evidence_reader.py': '95a3e3b9956bae7b998cd528c1e8e3023eda3a0a4f006ef29ca224eb8337b03c', 'official_manifest.json': 'd013e1633940c27d708e2474a8280990000e33018e889ed14655c856329ff1d8', 'web_runner.py': '4b879032a2f8e55df581cae6420282e9534105726d969a9c21696dab665ac549', 'candidate_profile.json': '6a71689859207cd78075703fb6717d358f53ddb502abb0d6c533cb796221eacf', 'full_m_compact_storage_single_v1.zip': '4fb83e4984c6ea230cc26f07563315c71667a68dc431ad2cdda89459e088a905', 'full_m_compact_storage_single_v1_manifest.json': '46794e01c4771c28014a204af5e07f2f659d8bff7a1054c42f333b45e8ddd7fc'}
STAGED = Path(tempfile.mkdtemp(prefix='soup-web-'+MODE+'-', dir='/kaggle/working'))
for name, encoded in PAYLOAD.items():
    raw = base64.b64decode(encoded, validate=True)
    assert hashlib.sha256(raw).hexdigest() == PAYLOAD_SHA256[name], name
    (STAGED/name).write_bytes(raw)
print('Prepared mode:', MODE)
print('Official source will be verified by bytes, not .git.')
print('Requested native training cap: 80 seconds; no external-cap bypass.')


## Execute the authorized prepared-job screen
The notebook process imports no Torch and keeps no CUDA context. Short-lived probe/preflight children exit before idle measurement. Full task/LoRA/data/precision remain unchanged. Small evidence files export incrementally below SOUP_RESULTS; adapters and compiler caches stay local. An external whole-job kill can leave status RUNNING: that is censored, never success.


In [ ]:
import subprocess, sys, signal
# Executing/submitting this notebook is the GPU action. Preparation did not run it.
# Never force-kill this supervisor: its guardian owns descendant cleanup and lock.
process = subprocess.Popen([sys.executable, '-B', str(STAGED/'web_runner.py'), '--mode', MODE], start_new_session=True)
try:
    return_code = process.wait()
except BaseException:
    if process.poll() is None:
        process.send_signal(signal.SIGTERM)
    while process.poll() is None:
        try: process.wait()
        except KeyboardInterrupt: pass
    raise
if return_code:
    raise RuntimeError('Screen blocked, failed, or censored. Inspect exported status/logs; do not treat it as a score or automatically retry.')
